# NOTEBOOK 1: RF-DETR Single-Class Pipeline (Base Model)

**Purpose:** Train a high-performance **RF-DETR Base** model on **1 class only** across all annotated images.

### Pipeline Features
- **Sample Mode vs Full Data Mode**: Configured via `SAMPLE_SIZE = 1000` (for rapid flow testing) or `SAMPLE_SIZE = None` (for full training).
- **Dynamic Folder Creation**: Automatically creates `./single_class/` and `./model/` subdirectories.
- **Early Visual EDA**: Visualizes sample images with their *original multi-category* bounding boxes and distinct colors before 1-class training.
- **Optimizations**: Adam optimizer, Cosine LR decay, Smaller LR (`5e-5`), Resolution `560`, Early stopping (`patience=5`), GPU T4 batch size `2`, and NMS post-processing.


In [ ]:

# STEP 0 — Install Dependencies (RF-DETR 1.4.0 & CUDA 12.1 Stack)
# 1. PyTorch Stack with CUDA 12.1 (cu121)
!pip install torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

# 2. Strict Core Packages & Flexible Dependencies
!pip install "rfdetr==1.4.0" "supervision>=0.22.0" "pycocotools>=2.0.7" "pytorch-lightning>=2.5.0" "numpy>=1.24.0,<2.0.0" "pandas>=2.0.0" "scikit-learn>=1.3.0" "opencv-python>=4.8.0" "albumentations>=1.3.0" "transformers>=4.40.0" "timm>=0.9.0" "accelerate>=0.28.0" "roboflow>=1.3.0" "rf100vl>=1.1.0"


In [ ]:

# CELL 1 — Imports & Logger Initialization
import os
import re
import json
import math
import random
import shutil
import logging
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from dotenv import load_dotenv
import supervision as sv

from azure.storage.blob import BlobServiceClient

import torch
from rfdetr import RFDETRBase

# Setup structured logging
log_format = "%(asctime)s | %(levelname)-7s | %(message)s"
logging.basicConfig(level=logging.INFO, format=log_format)
logger = logging.getLogger("RFDETR_Notebook1")

logger.info(f"PyTorch Version: {torch.__version__}")
logger.info(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    logger.info(f"GPU Device: {torch.cuda.get_device_name(0)}")
logger.info(f"Supervision Version: {sv.__version__}")


In [ ]:

# CELL 2 — Configuration & Mode Setup
# ==============================================================================
# SAMPLE vs FULL DATA MODE:
# Set SAMPLE_SIZE = 1000 to sample 1,000 images for fast testing and flow verification.
# Set SAMPLE_SIZE = None to run on the complete full dataset.
# ==============================================================================
SAMPLE_SIZE = 1000  # Set to None for full dataset training

# Project Directories (Created automatically from this notebook)
PROJECT_NAME = "single_class"
MODE_TAG = f"sample_{SAMPLE_SIZE}" if SAMPLE_SIZE else "full_data"

INPUT_JSON_DIR = Path("./coco_files")
DATASET_DIR = Path(f"./{PROJECT_NAME}/dataset_{MODE_TAG}")
OUTPUT_DIR = Path(f"./{PROJECT_NAME}/output_{MODE_TAG}")
INFERENCE_OUTPUT_DIR = Path(f"./{PROJECT_NAME}/inference_{MODE_TAG}")
MODEL_DIR = Path(f"./model/{PROJECT_NAME}")

PRETRAINED_WEIGHTS = "/home/jupyter/rf-detr-base-coco.pth"
AZURE_CONNECTION_STRING_ENV = "AZURE_STORAGE_CONNECTION_STRING"

IMAGE_FIELD = "image_id"
CATEGORY_FIELD = "category_id"
BBOX_FIELD = "bbox"
BBOX_FORMAT = "xywh"

# SINGLE CLASS CONFIGURATION
TARGET_CLASS = "object"  # Unified label name (all annotations remapped to 0)
NUM_CLASSES = 1

# Model Resolution & Optimizer
RESOLUTION = 560
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OPTIMIZER = "adam"
LR_SCHEDULER = "cosine"
EPOCHS = 50
BATCH_SIZE = 2             # Safe batch size to prevent CUDA OOM on T4 GPUs
LR = 5e-5                  # Smaller learning rate for Adam fine-tuning
WEIGHT_DECAY = 1e-4        # Adam weight decay regularization
NUM_WORKERS = 2            # Dataloader workers
GRAD_ACCUM_STEPS = 1

# Early Stopping
EARLY_STOPPING = True
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 0.001

TRAIN_RATIO = 0.80
VALID_RATIO = 0.10
TEST_RATIO = 0.10
RANDOM_SEED = 42

DOWNLOAD_WORKERS = 16
CONFIDENCE = 0.50
NMS_THRESHOLD = 0.50
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")

logger.info(f"Configuration loaded for [{PROJECT_NAME.upper()}] in [{MODE_TAG.upper()}] mode (Resolution={RESOLUTION}, Adam, LR={LR}).")


In [ ]:

# CELL 3 — Dynamic Folder Creation from Notebook
for p in [DATASET_DIR, OUTPUT_DIR, INFERENCE_OUTPUT_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

file_handler = logging.FileHandler(OUTPUT_DIR / "pipeline.log", mode="a", encoding="utf-8")
file_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(file_handler)

load_dotenv()
connection_string = os.getenv(AZURE_CONNECTION_STRING_ENV)
blob_service_client = BlobServiceClient.from_connection_string(connection_string) if connection_string else None
if not blob_service_client:
    logger.warning(f"Azure connection string not found in {AZURE_CONNECTION_STRING_ENV}. Using local images if available.")
else:
    logger.info("Azure BlobServiceClient initialized successfully.")

logger.info(f"Created and verified folders:\n - Dataset: {DATASET_DIR}\n - Output: {OUTPUT_DIR}\n - Inference: {INFERENCE_OUTPUT_DIR}\n - Model: {MODEL_DIR}")


In [ ]:

# CELL 4 — Read all annotation JSON files from coco_files/
json_files = sorted(INPUT_JSON_DIR.glob("*.json"))
if not json_files:
    raise FileNotFoundError(f"No JSON files found in {INPUT_JSON_DIR.resolve()}")

records = []
with tqdm(total=len(json_files), desc="Reading annotation JSON files", unit="file") as pbar:
    for json_file in json_files:
        with json_file.open("r", encoding="utf-8") as f:
            content = f.read().strip()
        parsed = False
        try:
            data = json.loads(content)
            if isinstance(data, list):
                for item in data:
                    if isinstance(item, list):
                        records.extend(item)
                    elif isinstance(item, dict):
                        records.append(item)
            elif isinstance(data, dict):
                if "annotations" in data and isinstance(data["annotations"], list):
                    records.extend(data["annotations"])
                else:
                    records.append(data)
            parsed = True
        except Exception:
            parsed = False
        if not parsed:
            for line in content.splitlines():
                line = line.strip()
                if not line:
                    continue
                try:
                    item = json.loads(line)
                    if isinstance(item, list):
                        records.extend(item)
                    elif isinstance(item, dict):
                        records.append(item)
                except Exception:
                    pass
        pbar.update(1)

logger.info(f"Processed {len(json_files)} JSON file(s). Total raw records: {len(records)}")


In [ ]:

# CELL 5 — Parse Annotations & Record Original Categories for Visual EDA
def normalize_category(value):
    if isinstance(value, (list, tuple)):
        if len(value) == 0:
            return None
        value = value[0]
    if value is None:
        return None
    cleaned = str(value).strip()
    return cleaned if cleaned else None

original_annotations = []
skipped = 0
for rec in records:
    try:
        image_url = str(rec.get(IMAGE_FIELD, "")).strip()
        category = normalize_category(rec.get(CATEGORY_FIELD))
        bbox = rec.get(BBOX_FIELD)
        if not category or not image_url or not isinstance(bbox, (list, tuple)) or len(bbox) != 4:
            skipped += 1
            continue
        x, y, w, h = map(float, bbox)
        if w <= 0 or h <= 0:
            skipped += 1
            continue
        original_annotations.append({
            "image_url": image_url,
            "original_category": category,
            "bbox": [x, y, w, h],
        })
    except Exception:
        skipped += 1

ORIGINAL_CATEGORIES = sorted(list({ann["original_category"] for ann in original_annotations}))
orig_cat_counts = defaultdict(int)
for ann in original_annotations:
    orig_cat_counts[ann["original_category"]] += 1

logger.info(f"Kept annotations: {len(original_annotations)} | Skipped: {skipped}")
logger.info(f"Original Categories ({len(ORIGINAL_CATEGORIES)}): {ORIGINAL_CATEGORIES}")
logger.info(f"Original Category Distribution: {dict(orig_cat_counts)}")


In [ ]:

# CELL 6 — Group Annotations & Apply Sample 1000 / Full Data Mode
image_records = defaultdict(list)
for ann in original_annotations:
    image_records[ann["image_url"]].append({
        "bbox": ann["bbox"],
        "original_category": ann["original_category"],
    })

image_urls = sorted(image_records.keys())
total_available_images = len(image_urls)

if SAMPLE_SIZE is not None and total_available_images > SAMPLE_SIZE:
    rng = random.Random(RANDOM_SEED)
    sampled_urls = sorted(rng.sample(image_urls, SAMPLE_SIZE))
    image_records = defaultdict(list, {u: image_records[u] for u in sampled_urls})
    image_urls = sorted(image_records.keys())
    logger.info(f"SAMPLE MODE ACTIVE: Sampled {len(image_urls)} images out of {total_available_images} total.")
else:
    logger.info(f"FULL DATA MODE ACTIVE: Using all {len(image_urls)} images.")


In [ ]:

# CELL 7 — Azure URL Helpers & Download Function
def parse_blob_url(url):
    clean_url = url.split("?", 1)[0]
    match = re.match(r"https?://([^/]+)/(.+)", clean_url)
    if not match:
        raise ValueError(f"Invalid Azure Blob URL: {url}")
    account_host = match.group(1)
    blob_path = match.group(2)
    parts = blob_path.split("/", 1)
    if len(parts) != 2:
        raise ValueError(f"Invalid blob path: {blob_path}")
    return account_host, parts[0], parts[1]

def safe_filename_from_url(url, index):
    clean = url.split("?", 1)[0]
    name = Path(clean).name
    if not name or Path(name).suffix.lower() not in IMAGE_EXTENSIONS:
        name = f"image_{index:08d}.jpg"
    name = re.sub(r"[^A-Za-z0-9._-]", "_", name)
    return f"{index:08d}_{name}"

def download_one(args):
    index, url = args
    try:
        _, container_name, blob_name = parse_blob_url(url)
        blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
        filename = safe_filename_from_url(url, index)
        output_path = DATASET_DIR / "downloaded_images" / filename
        output_path.parent.mkdir(parents=True, exist_ok=True)
        if not output_path.exists():
            output_path.write_bytes(blob_client.download_blob().readall())
        with Image.open(output_path) as img:
            width, height = img.size
        return {"url": url, "path": str(output_path), "filename": filename, "width": width, "height": height, "error": None}
    except Exception as e:
        return {"url": url, "path": None, "filename": None, "width": None, "height": None, "error": str(e)}


In [ ]:

# CELL 8 — Parallel Azure Image Downloads
download_results = {}
if blob_service_client is not None:
    with tqdm(total=len(image_urls), desc="Downloading images from Azure", unit="image") as pbar:
        with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
            futures = {executor.submit(download_one, (i, url)): url for i, url in enumerate(image_urls)}
            for future in as_completed(futures):
                res = future.result()
                download_results[res["url"]] = res
                pbar.update(1)
    download_errors = {u: r["error"] for u, r in download_results.items() if r["error"]}
    logger.info(f"Successfully downloaded/verified: {len(download_results) - len(download_errors)}")
    if download_errors:
        logger.warning(f"Download errors: {len(download_errors)}. First error: {next(iter(download_errors.values()))}")
else:
    logger.info("Using local images directly (Azure download skipped).")


In [ ]:

# CELL 9 — Early Visual EDA: Print Images with BBoxes & Original Categories in the Beginning
valid_downloaded_urls = [u for u in image_urls if u in download_results and download_results[u]["error"] is None]
eda_color_palette = sv.ColorPalette.from_hex([
    "#e6194B", "#3cb44b", "#ffe119", "#4363d8", "#f58231",
    "#911eb4", "#42d4f4", "#f032e6", "#bfef45", "#fabed4", "#469990"
])
orig_cat_to_id = {cat: idx for idx, cat in enumerate(ORIGINAL_CATEGORIES)}

def display_eda_samples(sample_urls, max_samples=3):
    logger.info(f"Rendering Early Visual EDA for {min(len(sample_urls), max_samples)} sample images...")
    for url in sample_urls[:max_samples]:
        res = download_results[url]
        img_path = res["path"]
        if not img_path or not Path(img_path).exists():
            continue
        image = Image.open(img_path).convert("RGB")
        boxes_xyxy, labels, class_ids = [], [], []
        for item in image_records[url]:
            x, y, w, h = item["bbox"]
            boxes_xyxy.append([x, y, x + w, y + h])
            cat_name = item["original_category"]
            labels.append(cat_name)
            class_ids.append(orig_cat_to_id[cat_name])
        if not boxes_xyxy:
            continue
        detections = sv.Detections(xyxy=np.array(boxes_xyxy, dtype=np.float32), class_id=np.array(class_ids, dtype=int))
        text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)
        thickness = sv.calculate_optimal_line_thickness(resolution_wh=image.size)
        box_annotator = sv.BoxAnnotator(color=eda_color_palette, thickness=thickness)
        label_annotator = sv.LabelAnnotator(color=eda_color_palette, text_color=sv.Color.BLACK, text_scale=text_scale, smart_position=True)
        annotated_scene = box_annotator.annotate(scene=np.array(image), detections=detections)
        annotated_scene = label_annotator.annotate(scene=annotated_scene, detections=detections, labels=labels)
        plt.figure(figsize=(12, 8))
        plt.imshow(annotated_scene)
        plt.axis("off")
        plt.title(f"Early Visual EDA: {Path(img_path).name} | Categories: {set(labels)}", fontsize=11)
        plt.show()

if valid_downloaded_urls:
    display_eda_samples(valid_downloaded_urls, max_samples=3)


In [ ]:

# CELL 10 — Train/Val/Test Split for 1-Class Training Across All Images
valid_urls = [u for u in image_urls if u in download_results and download_results[u]["error"] is None]
rng = random.Random(RANDOM_SEED)
rng.shuffle(valid_urls)

n = len(valid_urls)
n_train = int(n * TRAIN_RATIO)
n_valid = int(n * VALID_RATIO)

SPLITS = {
    "train": valid_urls[:n_train],
    "val": valid_urls[n_train:n_train + n_valid],
    "test": valid_urls[n_train + n_valid:],
}
logger.info(f"Image counts per split: { {k: len(v) for k, v in SPLITS.items()} }")


In [ ]:

# CELL 11 — Create Split Folders & Copy Images
IMAGE_METADATA = {}
for split in SPLITS:
    (DATASET_DIR / split / "images").mkdir(parents=True, exist_ok=True)

valid_alias = DATASET_DIR / "valid"
if not valid_alias.exists():
    try:
        valid_alias.symlink_to("val", target_is_directory=True)
    except Exception:
        pass

with tqdm(total=sum(len(v) for v in SPLITS.values()), desc="Copying split images", unit="image") as pbar:
    for split, urls in SPLITS.items():
        for url in urls:
            res = download_results[url]
            src = Path(res["path"])
            dst = DATASET_DIR / split / "images" / res["filename"]
            if src.resolve() != dst.resolve() and not dst.exists():
                shutil.copy2(src, dst)
            IMAGE_METADATA[url] = {
                "filename": res["filename"],
                "width": int(res["width"]),
                "height": int(res["height"]),
                "path": str(dst),
            }
            pbar.update(1)
logger.info(f"Organized images in split directories. Total cached: {len(IMAGE_METADATA)}")


In [ ]:

# CELL 12 — Build Single-Class COCO Annotations (All Categories Remapped to Class 0)
def clip_xywh(x, y, w, h, width, height):
    x1 = max(0.0, min(float(x), float(width)))
    y1 = max(0.0, min(float(y), float(height)))
    x2 = max(0.0, min(float(x + w), float(width)))
    y2 = max(0.0, min(float(y + h), float(height)))
    return x1, y1, x2 - x1, y2 - y1

def build_single_class_coco(split, urls):
    images = []
    annotations = []
    url_to_image_id = {}
    for image_id, url in enumerate(urls, start=1):
        meta = IMAGE_METADATA[url]
        images.append({
            "id": image_id,
            "file_name": f"images/{meta['filename']}",
            "width": meta["width"],
            "height": meta["height"],
        })
        url_to_image_id[url] = image_id

    ann_id = 1
    for url in urls:
        meta = IMAGE_METADATA[url]
        image_id = url_to_image_id[url]
        for item in image_records[url]:
            x, y, w, h = item["bbox"]
            x, y, w, h = clip_xywh(x, y, w, h, meta["width"], meta["height"])
            if w <= 0 or h <= 0:
                continue
            annotations.append({
                "id": ann_id,
                "image_id": image_id,
                "category_id": 0,
                "bbox": [x, y, w, h],
                "area": w * h,
                "iscrowd": 0,
            })
            ann_id += 1

    return {
        "info": {"description": f"RF-DETR single-class ({TARGET_CLASS}) dataset ({MODE_TAG})"},
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": [{"id": 0, "name": TARGET_CLASS, "supercategory": "object"}],
    }

backup_dir = DATASET_DIR.parent / f"{DATASET_DIR.name}_backup_annotations"
backup_dir.mkdir(parents=True, exist_ok=True)
for split, urls in SPLITS.items():
    coco = build_single_class_coco(split, urls)
    output_json = DATASET_DIR / split / "_annotations.coco.json"
    with output_json.open("w", encoding="utf-8") as f:
        json.dump(coco, f, separators=(",", ":"))
    shutil.copy2(output_json, backup_dir / f"{split}_annotations.coco.json")
logger.info(f"Single-class COCO files generated and saved to: {DATASET_DIR}")


In [ ]:

# CELL 13 — Dataset Validation
dataset_summary = {}
for split in SPLITS:
    split_dir = DATASET_DIR / split
    with (split_dir / "_annotations.coco.json").open("r", encoding="utf-8") as f:
        coco = json.load(f)
    missing = [img["file_name"] for img in coco["images"] if not (split_dir / img["file_name"]).exists()]
    dataset_summary[split] = {"images": len(coco["images"]), "annotations": len(coco["annotations"]), "missing": len(missing)}
    if missing:
        raise FileNotFoundError(f"{split}: {len(missing)} image files missing!")
logger.info(f"Dataset validation passed: {json.dumps(dataset_summary, indent=2)}")


In [ ]:

# CELL 14 — Verify Pretrained Weights
weights_path = Path(PRETRAINED_WEIGHTS)
if not weights_path.exists():
    logger.warning(f"Pretrained weights not found at {weights_path}. Model will download official weights if needed.")
else:
    logger.info(f"Verified local pretrained weights: {weights_path}")


In [ ]:

# CELL 15 — Initialize RF-DETR Base Model (Resolution 560, 1 Class)
model = RFDETRBase(
    pretrain_weights=PRETRAINED_WEIGHTS if Path(PRETRAINED_WEIGHTS).exists() else "rf-detr-base-coco.pth",
    resolution=RESOLUTION,
    device=DEVICE,
    num_classes=1
)
logger.info(f"RF-DETR Base model initialized with 1 class ({TARGET_CLASS}) at resolution={RESOLUTION} on {DEVICE}.")


In [ ]:

# CELL 16 — Training Configuration with Adam, Smaller LR 5e-5, Cosine Decay & Resolution 560
train_kwargs = {
    "dataset_dir": str(DATASET_DIR),
    "train_split": "train",
    "val_split": "val",
    "annotation_file": "_annotations.coco.json",
    "image_folder": "images",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "optimizer": OPTIMIZER,
    "lr_scheduler": LR_SCHEDULER,
    "num_workers": NUM_WORKERS,
    "output_dir": str(OUTPUT_DIR),
    "early_stopping": EARLY_STOPPING,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
}
logger.info(f"Training Arguments:\n{json.dumps(train_kwargs, indent=2)}")


In [ ]:

# CELL 17 — Train RF-DETR Base Model
logger.info("Starting RF-DETR Base training (Adam, Cosine Decay, Early Stopping)...")
try:
    model.train(**train_kwargs)
except TypeError as e:
    logger.warning(f"Direct kwargs notice: {e}. Retrying with core training parameters...")
    model.train(
        dataset_dir=str(DATASET_DIR),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        optimizer=OPTIMIZER,
        lr_scheduler=LR_SCHEDULER,
        num_workers=NUM_WORKERS,
        output_dir=str(OUTPUT_DIR),
    )
logger.info("Training completed successfully.")


In [ ]:

# CELL 18 — Select Best Checkpoint and Copy to model/ Directory
preferred_names = [
    "checkpoint_best_regular.pth",
    "checkpoint_best_total.pth",
    "checkpoint_best.pth",
    "best.pth",
    "checkpoint_best.pt",
    "best.pt",
]

checkpoint_candidates = []
for pattern in ["*.pth", "*.pt", "*.ckpt"]:
    checkpoint_candidates.extend(OUTPUT_DIR.rglob(pattern))
checkpoint_candidates = sorted(set(checkpoint_candidates), key=lambda p: p.stat().st_mtime, reverse=True)

BEST_CHECKPOINT = None
for name in preferred_names:
    matches = list(OUTPUT_DIR.rglob(name))
    if matches:
        BEST_CHECKPOINT = matches[0]
        break
if BEST_CHECKPOINT is None and checkpoint_candidates:
    BEST_CHECKPOINT = checkpoint_candidates[0]

if BEST_CHECKPOINT is None:
    raise FileNotFoundError("No trained checkpoint found in output directory.")

logger.info(f"Selected Best Checkpoint: {BEST_CHECKPOINT}")

# Automatically copy best checkpoint into model/ folder
best_model_dest = MODEL_DIR / f"best_model_{MODE_TAG}.pth"
shutil.copy2(BEST_CHECKPOINT, best_model_dest)
logger.info(f"Copied best checkpoint to model directory: {best_model_dest.resolve()}")


In [ ]:

# CELL 19 — Load Trained Model & Optimize for Inference
trained_model = RFDETRBase(
    pretrain_weights=str(BEST_CHECKPOINT),
    resolution=RESOLUTION,
    device=DEVICE,
    num_classes=1
)

try:
    trained_model.optimize_for_inference()
    logger.info("trained_model.optimize_for_inference() executed successfully.")
except Exception as e:
    logger.warning(f"optimize_for_inference notice: {e}")

logger.info("Trained 1-class model ready for test evaluation.")


In [ ]:

# CELL 20 — Test Inference with NMS Post-Processing & Supervision Annotations
annotated_dir = INFERENCE_OUTPUT_DIR / "annotated_images"
annotated_dir.mkdir(parents=True, exist_ok=True)

test_image_paths = [Path(IMAGE_METADATA[u]["path"]) for u in SPLITS["test"]]
prediction_records = []
color_palette = sv.ColorPalette.from_hex(["#00ff00", "#3399ff", "#ff3366"])

with tqdm(total=len(test_image_paths), desc="Running test inference & NMS post-processing", unit="image") as pbar:
    for img_path in test_image_paths:
        image_pil = Image.open(img_path).convert("RGB")
        detections = trained_model.predict(str(img_path), threshold=CONFIDENCE)

        # NMS Post-Processing
        try:
            if hasattr(detections, "with_nms"):
                detections = detections.with_nms(threshold=NMS_THRESHOLD)
            elif hasattr(sv, "non_max_suppression"):
                detections = sv.non_max_suppression(detections, threshold=NMS_THRESHOLD)
        except Exception:
            pass

        text_scale = sv.calculate_optimal_text_scale(resolution_wh=image_pil.size)
        thickness = sv.calculate_optimal_line_thickness(resolution_wh=image_pil.size)

        box_annotator = sv.BoxAnnotator(color=color_palette, thickness=thickness)
        label_annotator = sv.LabelAnnotator(
            color=color_palette,
            text_color=sv.Color.BLACK,
            text_scale=text_scale,
            smart_position=True
        )

        labels = []
        if hasattr(detections, "confidence") and detections.confidence is not None:
            labels = [f"{TARGET_CLASS} {conf:.2f}" for conf in detections.confidence]

        annotated_frame = np.array(image_pil)
        if hasattr(detections, "xyxy") and len(detections.xyxy) > 0:
            annotated_frame = box_annotator.annotate(scene=annotated_frame, detections=detections)
            if labels:
                annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)

        annotated_img = Image.fromarray(annotated_frame)
        annotated_save_path = annotated_dir / img_path.name
        annotated_img.save(annotated_save_path)

        prediction_records.append({
            "image": str(img_path),
            "annotated_image": str(annotated_save_path),
            "prediction": detections,
        })
        pbar.update(1)

logger.info(f"Inference completed for {len(prediction_records)} test images. Saved to {annotated_dir.resolve()}")


In [ ]:

# CELL 21 — Save Prediction JSON
def make_json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if hasattr(obj, "tolist"):
        try:
            return obj.tolist()
        except Exception:
            pass
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)

predictions_json = INFERENCE_OUTPUT_DIR / "test_predictions.json"
with predictions_json.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(prediction_records), f, indent=2)
logger.info(f"Saved prediction results to: {predictions_json.resolve()}")


In [ ]:

# CELL 22 — Visual Preview of Predictions
def visualize_prediction(image_path):
    img_path = Path(image_path)
    annotated_path = INFERENCE_OUTPUT_DIR / "annotated_images" / img_path.name
    target = annotated_path if annotated_path.exists() else img_path
    image = Image.open(target).convert("RGB")
    plt.figure(figsize=(12, 8))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"RF-DETR 1-Class Prediction: {img_path.name}")
    plt.show()

for p in test_image_paths[:3]:
    visualize_prediction(p)


In [ ]:

# CELL 23 — Final Execution Summary
summary = {
    "pipeline": "Single-Class RF-DETR Base",
    "mode": MODE_TAG,
    "sample_size": SAMPLE_SIZE,
    "target_class": TARGET_CLASS,
    "num_classes": NUM_CLASSES,
    "resolution": RESOLUTION,
    "device": DEVICE,
    "optimizations": {
        "optimizer": OPTIMIZER,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "lr_scheduler": LR_SCHEDULER,
        "cudnn_benchmark": True,
        "float32_matmul_precision": "high",
        "batch_size": BATCH_SIZE,
        "early_stopping": EARLY_STOPPING,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "optimize_for_inference": True,
        "confidence_threshold": CONFIDENCE,
        "nms_threshold": NMS_THRESHOLD,
    },
    "train_images": len(SPLITS["train"]),
    "val_images": len(SPLITS["val"]),
    "test_images": len(SPLITS["test"]),
    "dataset_dir": str(DATASET_DIR),
    "output_dir": str(OUTPUT_DIR),
    "best_checkpoint": str(BEST_CHECKPOINT),
    "copied_model_path": str(best_model_dest),
    "predictions_file": str(predictions_json),
    "annotated_images_dir": str(annotated_dir),
}
print(json.dumps(summary, indent=2))
logger.info(f"Notebook 1 pipeline finished successfully. Model checkpoint ready at: {best_model_dest}")
